# VadCLIP V6 — Adapter thời gian đa tỉ lệ

**Ba cấu hình.** Control là VadCLIP huấn luyện lại với loss gốc. Single-scale dùng ba nhóm kênh có dilation (1, 1, 1). V6 dùng cùng adapter với dilation (1, 2, 4). Hai adapter có số tham số và khởi tạo giống nhau, với đầu ra residual khởi tạo bằng 0. Adapter xử lý CLIP feature trước VadCLIP, mask các vị trí padding và không trộn giữa video hoặc các block inference. Mỗi bộ dùng cùng feature, split, batch, seed, số epoch và tiêu chí chọn checkpoint cho cả ba cấu hình.

## Add Input trên Kaggle

| Dataset | File trong Dataset | Vai trò |
|---|---|---|
| VadCLIP UCF-Crime Train1610 V5 | vadclip_ucf_train_v5.zip | 1.610 video train gốc, CLIP feature crop __5.npy. Tách train/dev từ gói này. |
| VadCLIP XD-Violence Train3954 V5 | vadclip_xd_train_v5.zip | 3.954 video train gốc, CLIP feature crop __0.npy. Lọc nhóm tên trùng test trước khi tách train/dev. |
| VadCLIP UCF-Crime Test 290 | ucfclipfeatures_test290.zip | CLIP feature của 290 video official test. |
| VadCLIP UCF-Crime Checkpoint | vadclip_ucf_checkpoint_upload.zip hoặc model_ucf.pth | Checkpoint V0 để kiểm fingerprint gói UCF. |
| VadCLIP XD-Violence Test 800 V2 | vadclip_xd_test800_kaggle_v2.zip | CLIP feature của 800 video official test, bản đổi tên an toàn có manifest V2. |
| VadCLIP XD-Violence Checkpoint | vadclip_xd_checkpoint.zip hoặc model_xd.pth | Checkpoint V0 để kiểm fingerprint gói XD. |

Tên Dataset có thể khác trong tài khoản. Notebook dò bằng tên file hoặc manifest. Gắn sáu Dataset V5 hiện có. Không cần video gốc, file kết quả V5 hay checkpoint V5. Notebook lấy CSV và ground truth từ commit VadCLIP đã khóa. OpenAI CLIP ViT-B/16 tải tự động hoặc đặt CLIP_WEIGHTS đến file ViT-B-16.pt chính thức đã gắn sẵn. Bật Internet để lấy mã upstream và gói phụ thuộc.

## Chạy

1. Bật GPU T4 và Internet. Giữ RUN_MODE = "smoke" rồi Run All. Smoke thử 2 batch mỗi cấu hình, dev nhỏ, nạp lại checkpoint và thử inference trên mỗi benchmark.
2. Khi smoke đạt trên hai bộ, đổi RUN_MODE = "full" rồi Run All. DATASETS mặc định chạy cả hai. Có thể đặt ("ucf",) hoặc ("xd",) để chạy riêng khi cần.
3. Xem comparison.csv, summary.json, training.json, split_manifest.json, run.json và predictions.npz trong /kaggle/working/vadclip_v6/run_*/. V0 chỉ là mốc công bố. Chênh V6 với single-scale đo lợi ích riêng của dilation đa tỉ lệ.

**Giao thức.** Tách development từ official train và chọn checkpoint theo AP cấp video (top-k) của nhánh chính. UCF dùng C-Branch, XD dùng A-Branch. XD loại clip train có tiền tố tên nguồn trùng test theo quy tắc tách __#, chỉ dùng ID, không dùng nhãn test. Đây là phép loại theo tên, chưa xác minh danh tính nguồn clip. Lưu tất cả clip bị loại trong manifest và giữ nguyên hai official test. Báo thêm chẩn đoán điểm đỉnh theo khoảng sự kiện <=48, 49–160 và >160 frame. Chẩn đoán dùng nhãn frame khi chấm và không tham gia chọn mô hình. Phép chạy này chấm toàn bộ benchmark, không chấm tập con bốn hành vi.

Nguồn mã: [VadCLIP](https://github.com/nwpu-zxr/VadCLIP/tree/c41067f07d252efcda18008bea367886070c33b0). Hướng đa tỉ lệ tham khảo [TE-VTAF](https://www.mdpi.com/2313-433X/12/6/249).

## Chạy tiếp khi còn checkpoint từ lượt full

Trong phiên Kaggle có thư mục của lượt cũ, đặt RUN_MODE = "full" và
RESUME_FROM = "/kaggle/working/vadclip_v6/run_vhd52gjd". Giữ DATASETS = ("ucf", "xd")
để tổng hợp cả hai. Notebook dùng lại ba kết quả UCF và XD control đã có,
chỉ chạy evaluator cho các cấu hình còn thiếu, rồi ghi bảng mới trong run_* mới.
Nếu đang ở kernel của notebook cũ, thay ô cấu hình, ô ngay dưới tiêu đề
**Adapter V6 và trainer**, và ô cuối bằng bản cập nhật. Chạy lại ba ô đó theo thứ tự.
Nếu đã mất phiên Kaggle, cần khôi phục thư mục đầu ra đã tải xuống, gồm training.json,
checkpoint cần chấm và các run.json đã hoàn tất, rồi trỏ RESUME_FROM đến thư mục đó.


In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUN_MODE = "smoke"
RESUME_FROM = None  # Đặt đường dẫn run_* cũ để chạy tiếp phần chấm còn thiếu.
SEED = 42
EPOCHS = 4
HALF_BATCH = 8
DATASETS = ("ucf", "xd")
VARIANTS = {"control": (), "single_scale": (1, 1, 1), "v6": (1, 2, 4)}

# None: tìm qua Add Input. Điền đường dẫn nếu có nhiều gói trùng tên.
UCF_TRAIN_ROOT = None
XD_TRAIN_ROOT = None
UCF_DATA_ROOT = None
UCF_CHECKPOINT = None
XD_DATA_ROOT = None
XD_FEATURE_ZIP = None
XD_CHECKPOINT = None
CLIP_WEIGHTS = None

TRAIN_MANIFESTS = {
    "ucf": "08503ee0733dc34ca9b955ad8d7656d4de70fbe4903f1a716ca6aff56860683f",
    "xd": "156c0638d4668c4ef7896216108981e6d14cc582637ddad744a22db3718cf157",
}


## Adapter V6 và trainer


In [ ]:
SUPPORT_SOURCE = r'''
import numpy as np

ADAPTER_SOURCE = """
import torch
from torch import nn


class TemporalAdapter(nn.Module):
    # features [B,T,512], lengths [B] in snippets. Output has the same shape.
    def __init__(self, dilations, width=512, hidden=96):
        super().__init__()
        channels = hidden // len(dilations)
        self.reduce = nn.Linear(width, hidden)
        self.branches = nn.ModuleList(
            nn.Conv1d(channels, channels, 3, padding=d, dilation=d, groups=channels)
            for d in dilations
        )
        self.activation = nn.GELU()
        self.project = nn.Linear(hidden, width)
        nn.init.zeros_(self.project.weight)
        nn.init.zeros_(self.project.bias)

    def forward(self, features, lengths):
        lengths = torch.as_tensor(lengths, device=features.device)
        valid = torch.arange(features.shape[1], device=features.device)[None, :] < lengths[:, None]
        valid = valid.unsqueeze(-1)
        # The upstream XD evaluator passes half-precision features to an FP32 model.
        adapter_input = features.to(dtype=self.reduce.weight.dtype)
        hidden = self.reduce(adapter_input).masked_fill(~valid, 0).transpose(1, 2)
        parts = hidden.chunk(len(self.branches), dim=1)
        fused = torch.cat([branch(part) for branch, part in zip(self.branches, parts)], dim=1)
        delta = self.project(self.activation(fused.transpose(1, 2)))
        adapted = adapter_input + delta.masked_fill(~valid, 0)
        return adapted.to(dtype=features.dtype)


def add_adapter(base_model, dilations):
    if not dilations:
        return base_model

    class AdaptedVadCLIP(base_model):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            with torch.random.fork_rng(devices=[]):
                self.temporal_adapter = TemporalAdapter(dilations)

        def forward(self, images, padding_mask, text, lengths):
            return super().forward(self.temporal_adapter(images, lengths), padding_mask, text, lengths)

    return AdaptedVadCLIP
"""


def install_adapter(runtime, dilations):
    (runtime / "v6_adapter.py").write_text(ADAPTER_SOURCE, encoding="utf-8")
    with (runtime / "model.py").open("a", encoding="utf-8") as stream:
        stream.write("\nfrom v6_adapter import add_adapter\n")
        stream.write(f"CLIPVAD = add_adapter(CLIPVAD, {dilations!r})\n")


def duration_diagnostics(gt, snippet_scores, steps, frames_per_feature):
    frame_scores = np.repeat(snippet_scores, frames_per_feature)
    if sum(steps) != len(snippet_scores) or len(frame_scores) != len(gt):
        raise ValueError("Số điểm hoặc nhãn lệch khi phân tích độ dài.")
    buckets = {
        key: {"peaks": [], "background_margins": []}
        for key in ("1-48", "49-160", "161+")
    }
    offset = 0
    for size in steps:
        end = offset + size * frames_per_feature
        labels = np.asarray(gt[offset:end], dtype=bool)
        scores = frame_scores[offset:end]
        offset = end
        boundaries = np.flatnonzero(np.diff(np.r_[False, labels, False]))
        background = scores[~labels]
        background_p95 = float(np.quantile(background, 0.95)) if len(background) else None
        for start, stop in boundaries.reshape(-1, 2):
            length = stop - start
            key = "1-48" if length <= 48 else "49-160" if length <= 160 else "161+"
            peak = float(scores[start:stop].max())
            buckets[key]["peaks"].append(peak)
            if background_p95 is not None:
                buckets[key]["background_margins"].append(peak - background_p95)
    return {
        key: {
            "event_count": len(values["peaks"]),
            "event_count_with_background": len(values["background_margins"]),
            "mean_peak": float(np.mean(values["peaks"])) if values["peaks"] else None,
            "mean_peak_minus_video_background_p95": (
                float(np.mean(values["background_margins"])) if values["background_margins"] else None
            ),
        }
        for key, values in buckets.items()
    }


'''

TRAINER_SOURCE = r'''
import csv
import hashlib
import importlib
import io
import json
import random
import subprocess
import sys
import tempfile
import time
from pathlib import PurePosixPath
from shutil import copyfile, copytree
from zipfile import ZipFile

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit

from v6_runtime_support import install_adapter

COMMIT = "c41067f07d252efcda18008bea367886070c33b0"
CLIP_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
UCF_CLASSES = (
    "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",
    "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",
)
XD_LABELS = {
    "A": "normal", "B1": "fighting", "B2": "shooting", "B4": "riot",
    "B5": "abuse", "B6": "car accident", "G": "explosion",
}


def file_sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def source_basename(path):
    return PurePosixPath(str(path).replace(chr(92), "/")).name


def video_id(path):
    stem, sep, crop = source_basename(path).rpartition("__")
    if not stem or not sep or crop not in {f"{i}.npy" for i in range(10)}:
        raise ValueError(f"Tên feature VadCLIP sai: {path}")
    return stem.casefold()


def checkout():
    if not SRC_REPO.exists():
        SRC_REPO.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as tmp:
            clone = Path(tmp) / "source"
            subprocess.run(["git", "clone", "https://github.com/nwpu-zxr/VadCLIP.git", str(clone)], check=True)
            subprocess.run(["git", "-C", str(clone), "checkout", "--detach", COMMIT], check=True)
            clone.rename(SRC_REPO)
    actual = subprocess.check_output(["git", "-C", str(SRC_REPO), "rev-parse", "HEAD"], text=True).strip()
    dirty = subprocess.check_output(
        ["git", "-C", str(SRC_REPO), "status", "--porcelain", "--untracked-files=all"], text=True,
    ).strip()
    if actual != COMMIT or dirty:
        raise RuntimeError("Cần checkout VadCLIP sạch tại commit đã khóa.")


def official_csv(default, repo):
    path = repo / "list" / source_basename(default)
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        if not reader.fieldnames or not {"path", "label"} <= set(reader.fieldnames):
            raise ValueError(f"Thiếu cột path/label: {path.name}")
        rows = list(reader)
    return rows, file_sha(path)


class FeaturePack:
    def __init__(self, root, prefix, expected_manifest):
        name = f"vadclip_{prefix}_train_v5"
        if root.is_file():
            candidates = [root]
        else:
            candidates = sorted(root.rglob(f"{name}_manifest.json"))
            if not candidates:
                candidates = sorted(root.rglob(f"{name}.zip"))
        if len(candidates) != 1:
            raise ValueError(f"Cần đúng một gói {name}.zip hoặc manifest giải nén. Tìm thấy {len(candidates)}.")
        location = candidates[0]
        self.archive = None
        self.directory = location.parent
        manifest_name = f"{name}_manifest.json"
        if location.suffix == ".zip":
            self.archive = ZipFile(location)
            payload = self.archive.read(manifest_name)
        else:
            payload = location.read_bytes()
        self.manifest_sha256 = hashlib.sha256(payload).hexdigest()
        if self.manifest_sha256 != expected_manifest:
            raise ValueError(f"Sai hash manifest gói {name}.")
        content = json.loads(payload)
        crop, count = (5, 1610) if prefix == "ucf" else (0, 3954)
        if (content["dataset"] != prefix or content["crop"] != crop
                or content["schema"] != "vadclip_v5_train_v1" or len(content["features"]) != count):
            raise ValueError(f"Sai schema hoặc số video trong gói {name}.")
        self.features = {}
        for idx, item in enumerate(content["features"], 1):
            if item["path"] != f"features/{prefix}_{idx:05d}.npy":
                raise ValueError("Tên vật lý trong manifest train không hợp lệ.")
            key = video_id(item["original"])
            if key in self.features:
                raise ValueError(f"ID trùng trong gói train: {key}")
            self.features[key] = item
        self.checked = set()

    def array(self, item):
        path = item["path"]
        data = self.archive.read(path) if self.archive else (self.directory / path).read_bytes()
        if path not in self.checked:
            if hashlib.sha256(data).hexdigest() != item["sha256"]:
                raise ValueError(f"SHA-256 sai cho feature train: {path}")
            self.checked.add(path)
        array = np.load(io.BytesIO(data), allow_pickle=False)
        if (array.shape != (item["steps"], 512) or not len(array)
                or array.dtype.kind != "f" or not np.isfinite(array).all()):
            raise ValueError(f"Shape/dtype/giá trị feature train sai: {path}")
        return array

    def close(self):
        if self.archive is not None:
            self.archive.close()


def select_train_rows(train_rows, features, encode_labels):
    """Gộp các crop của cùng một video, ưu tiên crop có trong gói feature."""
    selected = {}
    for row in train_rows:
        key = video_id(row["path"])
        feature = features.get(key)
        matches_pack_crop = (
            feature is not None
            and source_basename(row["path"]).casefold()
            == source_basename(feature["original"]).casefold()
        )
        previous = selected.get(key)
        if previous is not None:
            previous_row, previous_matches = previous
            if previous_row["label"] != row["label"]:
                encoded = encode_labels([previous_row["label"], row["label"]])
                if not torch.equal(encoded[0], encoded[1]):
                    raise ValueError(
                        f"Nhãn khác nhau giữa các crop của video {key}: "
                        f"{previous_row['label']} và {row['label']}."
                    )
            if previous_matches or not matches_pack_crop:
                continue
        selected[key] = (row, matches_pack_crop)
    return [row for row, _ in selected.values()]


def dataset_split(train_rows, test_rows, features, encode_labels, prefix, seed):
    train_ids = [video_id(row["path"]) for row in train_rows]
    test_ids = {video_id(row["path"]) for row in test_rows}
    if set(train_ids) & test_ids:
        raise ValueError("ID official train giao official test.")
    if set(train_ids) != features.keys():
        missing = set(train_ids) - features.keys()
        extra = features.keys() - set(train_ids)
        raise ValueError(f"Gói không khớp CSV train. Thiếu {len(missing)}, thừa {len(extra)}. Mẫu: {sorted(missing)[:4]}.")
    labels = encode_labels([row["label"] for row in train_rows]).float().cpu()
    if labels.ndim != 2 or (labels.sum(1) <= 0).any():
        raise ValueError("Nhãn video train có hàng rỗng.")
    abnormal = (labels[:, 0] == 0).numpy()
    if (labels[:, 0].bool() & labels[:, 1:].sum(1).bool()).any():
        raise ValueError("Nhãn Normal đồng thời có nhãn bất thường.")
    if (labels.sum(0) <= 0).any():
        raise ValueError("Official train thiếu lớp trong label map.")
    groups = [vid.split("__#", 1)[0] if prefix == "xd" else vid for vid in train_ids]
    test_groups = {vid.split("__#", 1)[0] if prefix == "xd" else vid for vid in test_ids}
    excluded = np.flatnonzero(np.isin(groups, sorted(test_groups)))
    eligible = np.flatnonzero(~np.isin(groups, sorted(test_groups)))
    if len(eligible) < 2 or (labels[eligible].sum(0) <= 0).any():
        raise ValueError("Lọc nhóm tên nguồn khiến train/dev thiếu dữ liệu hoặc thiếu lớp.")
    eligible_groups = np.asarray(groups)[eligible]
    candidates = GroupShuffleSplit(n_splits=30, test_size=0.10, random_state=seed)
    for train_positions, dev_positions in candidates.split(eligible, abnormal[eligible], eligible_groups):
        train, dev = eligible[train_positions], eligible[dev_positions]
        if (len(np.unique(abnormal[train])) == 2 and len(np.unique(abnormal[dev])) == 2
                and bool((labels[train].sum(0) > 0).all())):
            assert not set(np.asarray(groups)[train]) & set(np.asarray(groups)[dev])
            return train_ids, groups, labels, abnormal, train, dev, excluded, len(set(groups) & test_groups)
    raise ValueError("Chưa tìm được train/development đủ hai nhóm và độ phủ train.")


def balanced_batches(indices, abnormal, half, seed):
    rng = np.random.default_rng(seed)
    positive = rng.permutation(indices[abnormal[indices]])
    normal = rng.permutation(indices[~abnormal[indices]])
    batches = (max(len(positive), len(normal)) + half - 1) // half
    positive = np.resize(positive, batches * half).reshape(batches, half)
    normal = np.resize(normal, batches * half).reshape(batches, half)
    return np.concatenate((normal, positive), axis=1)


def make_batch(indices, pack, train_ids, labels, block_length, device):
    data = np.zeros((len(indices), block_length, 512), dtype=np.float32)
    lengths = []
    for slot, index in enumerate(indices):
        array = pack.array(pack.features[train_ids[index]])
        length = min(len(array), block_length)
        if len(array) > length:
            picks = np.linspace(0, len(array) - 1, length, dtype=np.int64)
            array = array[picks]
        data[slot, :length] = array
        lengths.append(length)
    return (
        torch.from_numpy(data).to(device),
        labels[indices].to(device),
        torch.tensor(lengths, dtype=torch.long, device=device),
        lengths,
    )


def bag_scores(logits_c, logits_a, lengths, primary):
    scores = logits_c.squeeze(-1).sigmoid() if primary == "C" else 1 - logits_a.softmax(dim=-1)[..., 0]
    if scores.ndim != 2 or len(scores) != len(lengths):
        raise ValueError("Shape logit VadCLIP sai khi tính điểm video.")
    return torch.stack([scores[i, :n].topk(n // 16 + 1).values.mean() for i, n in enumerate(lengths)])


def trainable_fingerprint(model, adapter):
    sha = hashlib.sha256()
    count = 0
    for name, param in model.named_parameters():
        if param.requires_grad and name.startswith("temporal_adapter.") == adapter:
            sha.update(name.encode())
            sha.update(param.detach().cpu().contiguous().numpy().tobytes())
            count += param.numel()
    return sha.hexdigest(), count


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def run_training():
    OUTPUT.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError("V6 cần Accelerator GPU trong Kaggle.")
    checkout()
    if CLIP_WEIGHTS is not None:
        source = Path(CLIP_WEIGHTS)
        if file_sha(source) != CLIP_SHA:
            raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")
        cache = Path.home() / ".cache" / "clip"
        cache.mkdir(parents=True, exist_ok=True)
        target = cache / "ViT-B-16.pt"
        if source.resolve() != target.resolve() and (
            not target.is_file() or file_sha(target) != CLIP_SHA
        ):
            with tempfile.TemporaryDirectory(dir=cache) as cache_tmp:
                pending = Path(cache_tmp) / target.name
                copyfile(source, pending)
                if file_sha(pending) != CLIP_SHA:
                    raise ValueError("Bản sao CLIP sai SHA-256.")
                pending.replace(target)
    packages = {"ftfy": "ftfy", "regex": "regex", "tqdm": "tqdm", "pandas": "pandas"}
    missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
    with tempfile.TemporaryDirectory(prefix="vadclip_v6_source_") as work:
        runtime = Path(work) / "src"
        copytree(SRC_REPO / "src", runtime)
        install_adapter(runtime, ())
        clip_file = runtime / "clip" / "clip.py"
        text = clip_file.read_text(encoding="utf-8")
        old = "from pkg_resources import packaging"
        if text.count(old) != 1:
            raise ValueError("Mã CLIP upstream khác commit đã khóa.")
        clip_file.write_text(text.replace(old, "import packaging.version"), encoding="utf-8")
        sys.path.insert(0, str(runtime))
        from clip.model import CLIP
        from model import CLIPVAD
        from utils.tools import get_batch_label, get_prompt_text
        from v6_adapter import add_adapter

        args = importlib.import_module(f"{PREFIX}_option").parser.parse_args([])
        source_train = importlib.import_module(f"{PREFIX}_train")
        train_rows, train_sha = official_csv(args.train_list, SRC_REPO)
        test_rows, test_sha = official_csv(args.test_list, SRC_REPO)
        label_map = {name: name.lower() for name in UCF_CLASSES} if PREFIX == "ucf" else XD_LABELS
        if PREFIX == "ucf":
            label_map["RoadAccidents"] = "roadAccidents"
        prompts = get_prompt_text(label_map)
        if len(prompts) != args.classes_num or prompts[0].casefold() != "normal":
            raise ValueError(f"Thứ tự lớp VadCLIP sai: {prompts}")
        primary = "C" if PREFIX == "ucf" else "A"
        contrast_weight = 0.1 if PREFIX == "ucf" else 0.0001

        pack = FeaturePack(Path(TRAIN_ROOT) if TRAIN_ROOT else INPUT, PREFIX, MANIFEST_SHA256)
        try:
            def encode_labels(labels):
                return get_batch_label(labels, prompts, label_map)

            train_csv_rows = len(train_rows)
            train_rows = select_train_rows(train_rows, pack.features, encode_labels)
            print(
                f"{PREFIX}: CSV train có {train_csv_rows:,} dòng, "
                f"gộp thành {len(train_rows):,} video theo gói feature.",
                flush=True,
            )
            split = dataset_split(train_rows, test_rows, pack.features, encode_labels, PREFIX, SEED)
            train_ids, groups, labels, abnormal, train, dev, excluded, source_overlap = split
            if labels.shape != (len(train_rows), len(prompts)):
                raise ValueError("Kích thước ma trận nhãn train sai.")
            manifest = [
                {"id": train_ids[i], "group": groups[i], "label": train_rows[i]["label"],
                 "split": name, "sha256": pack.features[train_ids[i]]["sha256"]}
                for name, indices in (("train", train), ("development", dev), ("excluded_source_name", excluded))
                for i in indices
            ]
            manifest_path = OUTPUT / "split_manifest.json"
            manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
            print(f"{PREFIX}: train={len(train)}, development={len(dev)}, "
                  f"official test={len(test_rows)}, nhóm tên nguồn trùng trước lọc={source_overlap}, "
                  f"clip loại={len(excluded)}", flush=True)
            if SMOKE_ONLY:
                dev = np.r_[dev[~abnormal[dev]][:2], dev[abnormal[dev]][:2]]
            histories = {}
            initialization = None
            adapter_initialization = None
            for variant, dilations in VARIANTS.items():
                started = time.perf_counter()
                torch.cuda.reset_peak_memory_stats(device)
                seed_all(SEED)
                model = add_adapter(CLIPVAD, dilations)(
                    args.classes_num, args.embed_dim, args.visual_length, args.visual_width,
                    args.visual_head, args.visual_layers, args.attn_window,
                    args.prompt_prefix, args.prompt_postfix, device,
                ).to(device)
                clip = [module for module in model.modules() if isinstance(module, CLIP)]
                if len(clip) != 1:
                    raise RuntimeError("Cần đúng một backbone OpenAI CLIP.")
                clip[0].requires_grad_(False).eval()
                digest, base_params = trainable_fingerprint(model, adapter=False)
                adapter_digest, adapter_params = trainable_fingerprint(model, adapter=True)
                if initialization is not None and digest != initialization:
                    raise RuntimeError("Khởi tạo VadCLIP khác đối chứng.")
                initialization = digest
                if dilations:
                    current = (adapter_digest, adapter_params)
                    if adapter_initialization is not None and current != adapter_initialization:
                        raise RuntimeError("Hai adapter phải có cùng khởi tạo và số tham số.")
                    adapter_initialization = current
                optimizer = torch.optim.AdamW(
                    [p for p in model.parameters() if p.requires_grad], lr=args.lr,
                )
                scheduler = torch.optim.lr_scheduler.MultiStepLR(
                    optimizer, args.scheduler_milestones, args.scheduler_rate,
                )
                history = []
                best_ap = -1.0
                for epoch in range(1 if SMOKE_ONLY else EPOCHS):
                    model.train()
                    clip[0].eval()
                    batches = balanced_batches(train, abnormal, 2 if SMOKE_ONLY else HALF_BATCH, SEED + epoch)
                    if SMOKE_ONLY:
                        batches = batches[:2]
                    total_loss = 0.0
                    for step, batch in enumerate(batches, 1):
                        x, y, lengths, sizes = make_batch(batch, pack, train_ids, labels, args.visual_length, device)
                        optimizer.zero_grad(set_to_none=True)
                        text_features, logits_c, logits_a = model(x, None, prompts, lengths)
                        normal_text = F.normalize(text_features, dim=-1)
                        contrast = (normal_text[1:] @ normal_text[0]).abs().mean()
                        upstream_loss = (
                            source_train.CLAS2(logits_c, y, lengths, device)
                            + source_train.CLASM(logits_a, y, lengths, device)
                            + contrast_weight * contrast
                        )
                        loss = upstream_loss
                        if not torch.isfinite(loss).item():
                            raise ValueError(f"Loss không hữu hạn: {variant} epoch={epoch + 1} step={step}")
                        loss.backward()
                        optimizer.step()
                        total_loss += loss.item()
                        if step % 50 == 0:
                            print(f"{PREFIX}/{variant} epoch={epoch+1} batch={step}/{len(batches)}", flush=True)
                    model.eval()
                    dev_scores = []
                    with torch.no_grad():
                        for first in range(0, len(dev), 2 * HALF_BATCH):
                            batch = dev[first:first + 2 * HALF_BATCH]
                            x, _, lengths, sizes = make_batch(batch, pack, train_ids, labels, args.visual_length, device)
                            _, logits_c, logits_a = model(x, None, prompts, lengths)
                            dev_scores.extend(bag_scores(logits_c, logits_a, sizes, primary).cpu().tolist())
                    dev_ap = float(average_precision_score(abnormal[dev], dev_scores))
                    history.append(dict(
                        epoch=epoch + 1, steps=len(batches), dev_video_ap=dev_ap,
                        base_loss=total_loss / len(batches),
                        learning_rate=optimizer.param_groups[0]["lr"],
                    ))
                    print(f"{PREFIX}/{variant}: {history[-1]}", flush=True)
                    if dev_ap > best_ap:
                        best_ap = dev_ap
                        best_epoch = epoch + 1
                        path = OUTPUT / f"{variant}.pth"
                        pending = path.with_suffix(".pth.partial")
                        torch.save(model.state_dict(), pending)
                        pending.replace(path)
                    scheduler.step()
                histories[variant] = {
                    "dilations": dilations, "best_dev_epoch": best_epoch,
                    "best_dev_video_ap": best_ap, "epochs": history,
                    "base_trainable_params": base_params, "adapter_trainable_params": adapter_params,
                    "adapter_initialization_sha256": adapter_digest if dilations else None,
                    "elapsed_seconds": time.perf_counter() - started,
                    "peak_torch_vram_mib": torch.cuda.max_memory_allocated(device) / 2**20,
                    "checkpoint_sha256": file_sha(OUTPUT / f"{variant}.pth"),
                }
                del optimizer, scheduler, model, clip
                torch.cuda.empty_cache()
        finally:
            pack.close()
    report = {
        "mode": "smoke" if SMOKE_ONLY else "full",
        "prefix": PREFIX, "commit": COMMIT, "seed": SEED, "primary_branch": primary,
        "max_epoch": 1 if SMOKE_ONLY else EPOCHS,
        "half_batch": 2 if SMOKE_ONLY else HALF_BATCH,
        "feature_manifest_sha256": MANIFEST_SHA256, "train_csv_sha256": train_sha,
        "test_csv_sha256": test_sha, "split_manifest_sha256": file_sha(manifest_path),
        "train_csv_row_count": train_csv_rows,
        "train_video_count": len(train_rows),
        "train_count": len(train), "development_count": len(dev),
        "source_name_groups_in_both_splits_before_filter": source_overlap,
        "excluded_source_name_clip_count": len(excluded),
        "trainable_initialization_sha256": initialization,
        "verified_feature_count": len(pack.checked), "variants": histories,
    }
    if not SMOKE_ONLY and len(pack.checked) != len(train_rows) - len(excluded):
        raise RuntimeError("Chưa đọc và kiểm hash toàn bộ feature train/development.")
    (OUTPUT / "training.json").write_text(json.dumps(report, indent=2), encoding="utf-8")


if __name__ == "__main__":
    run_training()
'''


## Hai evaluator giữ giao thức V0


In [ ]:
UCF_RUNNER = r'''
COMMIT = "c41067f07d252efcda18008bea367886070c33b0"
UPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"
EXPECTED_INPUTS = {
    "checkpoint_sha256": "d064aba05190ce5f6652151e6259b24ca6dda83a1e6289bf343afb6f905a2c79",
    "split_sha256": "d621f4ccfb3e26274d215ad53b83fd1f596bb7e9f85cde69e01399db3da97c7f",
    "ground_truth_sha256": "c6448801e0f73d19adf82506cd58d60a90c4f0913e49dcc1ad1d677ee12d6301",
    "annotation_sha256": "82d581cd4685480ffda23eb0fe1bca441b672bfe99c457adca2df4cf4780c503",
    "feature_inventory_sha256": "8aa3a3c58f9bbbcc8082e024f3892e014f059374e8b52ca737ae709945210026"
}

import csv
import hashlib
import importlib.util
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from collections import defaultdict
from shutil import copyfile, copyfileobj, copytree
from urllib.error import URLError
from zipfile import ZipFile

if not INPUT.is_dir():
    raise FileNotFoundError("Import notebook trên Kaggle và gắn input trước.")

def git(*args):
    return subprocess.check_output(["git", *map(str, args)], text=True).strip()


if not SRC_REPO.exists():
    SRC_REPO.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=SRC_REPO.parent) as checkout_dir:
        checkout = Path(checkout_dir) / "source"
        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)
        checkout.rename(SRC_REPO)
if git("-C", SRC_REPO, "rev-parse", "HEAD") != COMMIT:
    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} sai commit. Hãy dùng thư mục nguồn mới.")
if git("-C", SRC_REPO, "status", "--porcelain", "--untracked-files=no"):
    raise RuntimeError(f"Mã nguồn tại {SRC_REPO} có thay đổi. Hãy dùng checkout sạch.")

packages = {
    "ftfy": "ftfy", "packaging": "packaging", "regex": "regex",
    "tqdm": "tqdm", "sklearn": "scikit-learn",
}
missing_packages = [pkg for mod, pkg in packages.items() if importlib.util.find_spec(mod) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)

np = importlib.import_module("numpy")

# Modern setuptools no longer ships pkg_resources. Adapt the import in a copy.
stage_directory = tempfile.TemporaryDirectory(prefix="vadclip_ucf_v5_")
staging = Path(stage_directory.name)
runtime_src = staging / "runtime"
copytree(SRC_REPO / "src", runtime_src, dirs_exist_ok=True)
from v6_runtime_support import duration_diagnostics, install_adapter

install_adapter(runtime_src, DILATIONS)
clip_py = runtime_src / "clip" / "clip.py"
clip_source = clip_py.read_text(encoding="utf-8")
old_import = "from pkg_resources import packaging"
if clip_source.count(old_import) != 1:
    raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích. Kiểm commit.")
clip_py.write_text(clip_source.replace(old_import, "import packaging.version"), encoding="utf-8")

OUTPUT.mkdir(parents=True, exist_ok=True)
print("Commit VadCLIP", COMMIT)
print(f"Chấm UCF-Crime V6: {EVAL_CHECKPOINT}")

CLASSES = (
    "Normal", "Abuse", "Arrest", "Arson", "Assault", "Burglary", "Explosion",
    "Fighting", "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing", "Vandalism",
)
FRAMES_PER_FEATURE = 16
FEATURE_DIM = 512
VIDEO_COUNT = 290
CLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def read_split(list_root):
    """Đọc theo đúng thứ tự repo. Tính khoảng [start, end) theo frame gốc."""
    with (list_root / "ucf_CLIP_rgbtest.csv").open(newline="", encoding="utf-8") as stream:
        entries = list(csv.DictReader(stream))
    annotations = {}
    for line in (list_root / "Temporal_Anomaly_Annotation.txt").read_text().splitlines():
        if not line.strip():
            continue
        name, label, *bounds = line.split()
        video_id = Path(name).stem
        if len(bounds) != 4 or video_id in annotations:
            raise ValueError(f"Annotation không hợp lệ: {video_id}")
        intervals = []
        for start, end in zip(bounds[::2], bounds[1::2]):
            lo, hi = int(start), int(end)
            if (lo, hi) == (-1, -1):
                continue
            if not 0 <= lo < hi:
                raise ValueError(f"Khoảng frame không hợp lệ: {video_id} {(lo, hi)}")
            intervals.append((lo, hi))
        annotations[video_id] = (label, intervals)

    rows = []
    for entry in entries:
        feature_name = Path(entry["path"]).name
        match = re.fullmatch(r"(.+)__([0-9])\.npy", feature_name)
        if match is None:
            raise ValueError(f"Tên feature sai: {feature_name}")
        video_id, crop = match.groups()
        label, intervals = annotations[video_id]
        if label != entry["label"] or label not in CLASSES:
            raise ValueError(f"Nhãn không khớp giữa CSV và annotation: {video_id}")
        rows.append({
            "id": video_id, "label": label, "feature_name": feature_name,
            "crop": int(crop), "intervals": intervals,
        })
    if len(rows) != VIDEO_COUNT or len({row["id"] for row in rows}) != VIDEO_COUNT:
        raise ValueError("Cần đúng 290 ID duy nhất từ split UCF-Crime chính thức.")
    return rows



def stage_member(archive, member, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(member) as source, destination.open("wb") as target:
        copyfileobj(source, target, length=1024 * 1024)
    return destination


def find_inputs(root, keys):
    """Tìm theo tên file chính thức, kể cả khi Kaggle giữ ZIP nguyên gói."""
    found = defaultdict(list)
    if root.is_file():
        archive_path = root
    elif root.is_dir():
        for folder, _, filenames in os.walk(root):
            for name in filenames:
                key = name.casefold()
                if key in keys:
                    found[key].append(Path(folder) / name)
        if found:
            return found
        archives = sorted(root.rglob("ucfclipfeatures_test290.zip"))
        if len(archives) != 1:
            raise RuntimeError(
                "Gắn Dataset feature UCF-Crime của V0 hoặc chỉ định DATA_ROOT."
            )
        archive_path = archives[0]
    else:
        raise FileNotFoundError(f"Không thấy DATA_ROOT: {root}")

    with ZipFile(archive_path) as archive:
        for index, member in enumerate(archive.infolist()):
            name = Path(member.filename).name
            key = name.casefold()
            if not member.is_dir() and key in keys:
                destination = staging / "features" / str(index) / name
                found[key].append(stage_member(archive, member, destination))
    return found


def choose_checkpoint():
    if CHECKPOINT is not None:
        path = Path(CHECKPOINT)
        if not path.is_file():
            raise FileNotFoundError(f"Không thấy checkpoint: {path}")
    else:
        matches = sorted(INPUT.rglob("model_ucf.pth"))
        if not matches:
            matches = sorted(INPUT.rglob("vadclip_ucf_checkpoint_upload.zip"))
        if len(matches) != 1:
            raise RuntimeError("Gắn checkpoint UCF-Crime của V0 hoặc chỉ định CHECKPOINT.")
        path = matches[0]
    if path.suffix.lower() != ".zip":
        return path
    with ZipFile(path) as archive:
        members = [m for m in archive.infolist() if Path(m.filename).name == "model_ucf.pth"]
        if len(members) != 1:
            raise ValueError(f"ZIP checkpoint phải chứa đúng một model_ucf.pth: {path.name}")
        return stage_member(archive, members[0], staging / "checkpoint" / "model_ucf.pth")


def input_fingerprint(rows, list_root, checkpoint_path):
    inventory = [[row["id"], row["label"], row["steps"], row["feature_sha256"]] for row in rows]
    inventory_bytes = json.dumps(inventory, separators=(",", ":")).encode()
    return {
        "checkpoint_sha256": sha256(checkpoint_path),
        "split_sha256": sha256(list_root / "ucf_CLIP_rgbtest.csv"),
        "ground_truth_sha256": sha256(list_root / "gt_ucf.npy"),
        "annotation_sha256": sha256(list_root / "Temporal_Anomaly_Annotation.txt"),
        "feature_inventory_sha256": hashlib.sha256(inventory_bytes).hexdigest(),
    }


def prepare_clip_cache(source, cache_dir):
    """Đặt file OpenAI đã xác minh vào đúng cache mà loader gốc đọc."""
    if not source.is_file():
        raise FileNotFoundError(f"Không thấy CLIP_WEIGHTS: {source}")
    if sha256(source) != CLIP_SHA256:
        raise ValueError("CLIP_WEIGHTS sai SHA-256 của OpenAI ViT-B-16.pt.")
    target = cache_dir / "ViT-B-16.pt"
    if target.is_file() and sha256(target) == CLIP_SHA256:
        return target
    cache_dir.mkdir(parents=True, exist_ok=True)
    temporary = cache_dir / "ViT-B-16.pt.vadclip-part"
    copyfile(source, temporary)
    if sha256(temporary) != CLIP_SHA256:
        raise ValueError("Bản sao CLIP lỗi. Kiểm dung lượng và file input.")
    temporary.replace(target)
    return target


def feature_array(path):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    if array.ndim != 2 or array.shape[1] != FEATURE_DIM or array.shape[0] < 1:
        raise ValueError(f"Feature cần dạng [T, 512], nhận {array.shape}: {path}")
    if array.dtype not in (np.float16, np.float32, np.float64):
        raise ValueError(f"Feature phải là số thực, nhận {array.dtype}: {path}")
    return array


def frame_labels(row, length):
    labels = np.zeros(length, dtype=np.uint8)
    for start, end in row["intervals"]:
        labels[start:end] = 1
    return labels


def validate_ground_truth(rows, gt):
    """Kiểm cả nội dung và thứ tự, ngoài phép kiểm tổng số frame."""
    offset = 0
    for row in rows:
        stop = offset + row["steps"] * FRAMES_PER_FEATURE
        expected = frame_labels(row, stop - offset)
        if not np.array_equal(gt[offset:stop], expected):
            raise ValueError(
                f"Ground truth lệch tại {row['id']} (frame offset {offset}, "
                f"{row['steps']} bước). Kiểm lại bản video/feature và frame count."
            )
        offset = stop
    if offset != len(gt):
        raise ValueError(f"Số frame không khớp: input={offset}, ground truth={len(gt)}")

# Kiểm đầu vào và đối chiếu hash với V0 đã chạy.
list_root = SRC_REPO / "list"
rows = read_split(list_root)
data_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT
expected = {row["feature_name"].casefold(): row["feature_name"] for row in rows}
found = find_inputs(data_root, expected)
missing = sorted(expected[key] for key in expected if not found[key])
ambiguous = {
    expected[key]: [str(path) for path in sorted(found[key])]
    for key in expected if len(found[key]) > 1
}
if missing or ambiguous:
    issue_path = OUTPUT / "input_issues.json"
    issue_path.write_text(json.dumps({
        "missing": missing,
        "ambiguous": ambiguous,
    }, indent=2), encoding="utf-8")
    raise RuntimeError(
        f"Thiếu {len(missing)} file, trùng {len(ambiguous)} ID. "
        f"Xem {issue_path} và kiểm DATA_ROOT."
    )
for row in rows:
    path = found[row["feature_name"].casefold()][0]
    row["path"] = path
    row["steps"] = len(feature_array(path))
    row["feature_sha256"] = sha256(path)

gt = np.load(list_root / "gt_ucf.npy", allow_pickle=False)
if gt.ndim != 1 or set(np.unique(gt)) != {0.0, 1.0}:
    raise ValueError("Ground truth UCF cần vector nhị phân cấp frame.")
validate_ground_truth(rows, gt)
checkpoint_path = choose_checkpoint()
input_hashes = input_fingerprint(rows, list_root, checkpoint_path)
mismatches = [key for key, value in EXPECTED_INPUTS.items() if input_hashes[key] != value]
if mismatches:
    raise ValueError("Đầu vào khác lượt V0 đã đối chiếu: " + ", ".join(mismatches))
print(f"Đã khớp {len(rows)} video, {len(gt):,} frame và toàn bộ hash đầu vào với V0.")

import torch

sys.path.insert(0, str(runtime_src))
from model import CLIPVAD
from ucf_option import parser as ucf_parser
from utils.tools import get_batch_mask

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle. Mã VadCLIP gốc cần CUDA khi chạy forward.")
device = torch.device("cuda:0")
torch.cuda.set_device(device)
torch.manual_seed(234)

# Dùng đúng cấu hình UCF từ mã tác giả, không đọc cờ của kernel Jupyter.
args = ucf_parser.parse_args([])
evaluated_checkpoint = Path(EVAL_CHECKPOINT)
evaluated_sha256 = sha256(evaluated_checkpoint)
if evaluated_sha256 != EVAL_SHA256:
    raise ValueError("Checkpoint đã chọn khác hash trong training.json.")
state = torch.load(evaluated_checkpoint, map_location="cpu", weights_only=True)
if isinstance(state, dict):
    for field in ("state_dict", "model_state_dict"):
        if field in state:
            state = state[field]
            break
if CLIP_WEIGHTS is not None:
    prepare_clip_cache(Path(CLIP_WEIGHTS), Path.home() / ".cache" / "clip")
try:
    model = CLIPVAD(
        args.classes_num, args.embed_dim, args.visual_length, args.visual_width, args.visual_head,
        args.visual_layers, args.attn_window, args.prompt_prefix, args.prompt_postfix, device,
    )
except URLError as exc:
    raise RuntimeError(
        "Không tải được CLIP ViT-B/16. Bật Internet hoặc gắn ViT-B-16.pt chính thức "
        "và đặt CLIP_WEIGHTS bằng đường dẫn file."
    ) from exc
try:
    model.load_state_dict(state, strict=True)
except (RuntimeError, TypeError) as exc:
    raise RuntimeError(
        f"Checkpoint không khớp model VadCLIP UCF-Crime: {evaluated_checkpoint}. "
        "Kiểm lại file từ cột Model dòng UCF-Crime, khác file ViT-B-16.pt."
    ) from exc
del state
model.to(device).eval()
print("Đã nạp đầy đủ checkpoint trên", torch.cuda.get_device_name(device))

def predict_video(feature, model, device, block_length):
    """Trả điểm [T, 2], lần lượt sigmoid(C) và 1-softmax(A)[Normal]."""
    steps = len(feature)
    if steps < 1 or not np.isfinite(feature).all():
        raise ValueError("Feature rỗng hoặc chứa NaN/Inf.")
    blocks = (steps + block_length - 1) // block_length
    padded = np.zeros((blocks * block_length, FEATURE_DIM), dtype=np.float32)
    padded[:steps] = feature
    lengths = torch.tensor([
        min(block_length, steps - i * block_length) for i in range(blocks)
    ])
    visual = torch.from_numpy(padded.reshape(blocks, block_length, FEATURE_DIM)).to(device)
    mask = get_batch_mask(lengths, block_length).to(device)
    with torch.no_grad():
        _, logits_c, logits_a = model(visual, mask, list(CLASSES), lengths)
        c_score = logits_c.reshape(-1)[:steps].sigmoid()
        a_score = 1 - logits_a.reshape(-1, len(CLASSES))[:steps].softmax(dim=-1)[:, 0]
        scores = torch.stack((c_score, a_score), dim=1).cpu().numpy()
    if scores.shape != (steps, 2) or not np.isfinite(scores).all():
        raise ValueError(f"Điểm bất thường không hợp lệ: {scores.shape}")
    return scores


def run_video(row):
    feature = feature_array(row["path"])
    if feature.shape != (row["steps"], FEATURE_DIM):
        raise ValueError(f"Feature {row['id']} sai shape: {feature.shape}")
    return predict_video(feature, model, device, args.visual_length)


# Chọn theo thứ tự split và độ dài, trước khi xem điểm.
pilot = sorted({
    next(i for i, row in enumerate(rows) if row["label"] != "Normal"),
    next(i for i, row in enumerate(rows) if row["label"] == "Normal"),
    max(range(len(rows)), key=lambda i: rows[i]["steps"]),
})
torch.cuda.reset_peak_memory_stats(device)
started = time.perf_counter()
scores_by_index = {}
for i in pilot:
    before = time.perf_counter()
    scores_by_index[i] = run_video(rows[i])
    print(rows[i]["id"], scores_by_index[i].shape, f"{time.perf_counter() - before:.2f}s")
smoke_seconds = time.perf_counter() - started
print(f"Smoke test hoàn tất trong {smoke_seconds:.2f}s.")

if not SMOKE_ONLY:
    for i, row in enumerate(rows):
        if i not in scores_by_index:
            scores_by_index[i] = run_video(row)
        if (i + 1) % 25 == 0 or i + 1 == len(rows):
            print(f"Đã chạy {i + 1}/{len(rows)} video", flush=True)

from importlib.metadata import version

from sklearn.metrics import average_precision_score, roc_auc_score

SCORE_COLUMNS = ("classification", "alignment")




def score_benchmark(rows, snippet_scores, gt):
    steps_per_video = np.asarray([row["steps"] for row in rows])
    if snippet_scores.shape != (sum(steps_per_video), len(SCORE_COLUMNS)):
        raise ValueError("Số điểm snippet hoặc cột không khớp.")
    frame_scores = np.repeat(snippet_scores, FRAMES_PER_FEATURE, axis=0)
    if len(frame_scores) != len(gt):
        raise ValueError("Số điểm cấp frame không khớp ground truth.")
    frame_mask = np.repeat(
        [row["label"] != "Normal" for row in rows],
        steps_per_video * FRAMES_PER_FEATURE,
    )
    metrics = {}
    for column, name in enumerate(SCORE_COLUMNS):
        scores = frame_scores[:, column]
        metrics[name] = {
            "roc_auc": float(roc_auc_score(gt, scores)),
            "ap": float(average_precision_score(gt, scores)),
            "ano_auc": float(roc_auc_score(gt[frame_mask], scores[frame_mask])),
        }
    return metrics


selected = sorted(scores_by_index)
if not SMOKE_ONLY and selected != list(range(len(rows))):
    raise ValueError("Thiếu dự đoán. Cần chạy toàn bộ split trước khi chấm.")
selected_rows = [rows[i] for i in selected]
ordered = [scores_by_index[i] for i in selected]
if any(score.shape != (row["steps"], 2)
       for row, score in zip(selected_rows, ordered, strict=True)):
    raise ValueError("Số điểm từng video không khớp.")
steps_per_video = [row["steps"] for row in selected_rows]
scores = np.concatenate(ordered, axis=0)

output_dir = OUTPUT / ("smoke" if SMOKE_ONLY else "full")
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "run_kind": "smoke_inference" if SMOKE_ONLY else "full_checkpoint_inference",
    "benchmark_scope": "ucf_crime_official_test",
    "evaluation_status": "exploratory_on_previously_seen_official_test",
    "commit": COMMIT,
    "runtime_patch": "CLIP imports packaging.version directly",
    **input_hashes,
    "clip_backbone_sha256": CLIP_SHA256,
    "frames_per_feature": FRAMES_PER_FEATURE,
    "visual_block_length": args.visual_length,
    "label_order": CLASSES,
    "score_columns": SCORE_COLUMNS,
    "evaluated_checkpoint_sha256": evaluated_sha256,
    "python": sys.version.split()[0],
    "libraries": {name: version(name) for name in (
        "torch", "torchvision", "numpy", "scipy", "scikit-learn", "pillow",
    )},
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(device),
    "smoke_seconds": smoke_seconds,
    "inference_seconds": time.perf_counter() - started,
    "peak_torch_vram_mib": torch.cuda.max_memory_allocated(device) / (1024 ** 2),
    "video_count": len(selected_rows),
    "videos": [{key: value for key, value in row.items() if key != "path"}
               for row in selected_rows],
    "metrics": None,
}
if not SMOKE_ONLY:
    metrics = score_benchmark(rows, scores, gt)
    metadata["metrics"] = metrics
    metadata["duration_diagnostics"] = duration_diagnostics(gt, scores[:, 0], steps_per_video, FRAMES_PER_FEATURE)
    print(json.dumps(metrics, indent=2))

np.savez_compressed(
    output_dir / "predictions.npz",
    video_id=np.asarray([row["id"] for row in selected_rows]),
    video_label=np.asarray([row["label"] for row in selected_rows]),
    snippet_offsets=np.cumsum([0] + steps_per_video),
    score_columns=np.asarray(SCORE_COLUMNS),
    scores=scores,
)
(output_dir / "run.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
stage_directory.cleanup()
print("Đầu ra", output_dir)
'''


In [ ]:
XD_RUNNER = r'''
COMMIT = "c41067f07d252efcda18008bea367886070c33b0"
UPSTREAM_URL = "https://github.com/nwpu-zxr/VadCLIP.git"
VIDEO_COUNT = 800
FRAMES_PER_FEATURE = 16
FRAME_COUNT = 2_331_296
FEATURE_DIM = 512
PAPER_AP = 0.8451
PACK_MANIFEST_NAME = "xd_test_manifest.json"
PACK_ARCHIVE_NAME = "vadclip_xd_test800.zip"
PREVIOUS_PACK_ARCHIVE_NAME = "vadclip_xd_test800_kaggle_v2.zip"
PACK_MANIFEST_SHA256 = "8e5d329708857338fa94fb63a1d5733d9d8536250e6c09c3ba012f54dce997b2"
CLIP_SHA256 = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"

import csv
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import tempfile
from collections import defaultdict
from contextlib import ExitStack
from pathlib import PurePosixPath
from zipfile import BadZipFile, ZipFile

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("Evaluator XD cần chế độ smoke hoặc full.")
if not INPUT.is_dir():
    raise FileNotFoundError("Gắn dữ liệu qua Add Input trong Kaggle trước khi chạy.")

OUTPUT.mkdir(parents=True, exist_ok=True)
scratch = SRC_REPO.parent
scratch.mkdir(parents=True, exist_ok=True)
src_repo = SRC_REPO
if not src_repo.exists():
    with tempfile.TemporaryDirectory(dir=scratch) as checkout_dir:
        checkout = Path(checkout_dir) / "source"
        subprocess.run(["git", "clone", UPSTREAM_URL, str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", COMMIT], check=True)
        checkout.rename(src_repo)

def git(*args):
    return subprocess.check_output(["git", "-C", str(src_repo), *args], text=True).strip()

if git("rev-parse", "HEAD") != COMMIT or git("status", "--porcelain", "--untracked-files=all"):
    raise RuntimeError(f"Cần checkout sạch tại commit {COMMIT}. Kiểm tra {src_repo}.")

np = importlib.import_module("numpy")

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def zip_members(path):
    try:
        with ZipFile(path) as archive:
            return [name for name in archive.namelist() if not name.endswith("/")]
    except BadZipFile as exc:
        raise ValueError(f"ZIP chưa tải đủ hoặc bị hỏng: {path.name}") from exc


def basename(value):
    return PurePosixPath(str(value).replace(chr(92), "/")).name


def read_split(path):
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not {"path", "label"} <= set(reader.fieldnames):
            raise ValueError("CSV tác giả cần cột path và label.")
        fields, rows = reader.fieldnames, list(reader)
    source_paths = [row["path"] for row in rows]
    if (
        len(rows) != VIDEO_COUNT
        or len(set(source_paths)) != VIDEO_COUNT
        or any(not path or not basename(path).endswith("__0.npy") for path in source_paths)
        or any(not row["label"] for row in rows)
    ):
        raise ValueError(f"Split XD phải gồm {VIDEO_COUNT} dòng feature hợp lệ, duy nhất.")
    return fields, rows


def matching_files(root, expected):
    found = defaultdict(list)
    if FEATURE_ZIP is None:
        if not root.is_dir():
            raise FileNotFoundError(f"DATA_ROOT không tồn tại: {root}")
        for path in root.rglob("*.npy"):
            if path.name in expected:
                found[path.name].append((path, None))
        if found:
            return found
        archives = sorted(root.rglob("*.zip"))
    else:
        archives = [Path(FEATURE_ZIP)]

    for archive_path in archives:
        if not archive_path.is_file():
            raise FileNotFoundError(f"Không thấy FEATURE_ZIP: {archive_path}")
        for member in zip_members(archive_path):
            name = basename(member)
            if name in expected:
                found[name].append((archive_path, member))
    return found


def match_one(original, candidates):
    # Nếu hai file cùng tên, chỉ phân giải khi hậu tố thư mục chính thức khớp duy nhất.
    if len(candidates) <= 1:
        return candidates
    official = PurePosixPath(original.replace(chr(92), "/")).parts
    for depth in range(2, len(official) + 1):
        matches = [
            (path, member)
            for path, member in candidates
            if PurePosixPath((member or path.as_posix()).replace(chr(92), "/")).parts[-depth:]
            == official[-depth:]
        ]
        if len(matches) == 1:
            return matches
        if not matches:
            break
    return candidates


def pick_checkpoint(root):
    if CHECKPOINT is not None:
        path = Path(CHECKPOINT)
        if not path.is_file() or path.suffix.lower() not in {".pth", ".zip"}:
            raise FileNotFoundError(f"CHECKPOINT cần đường dẫn .pth hoặc .zip hợp lệ: {path}")
        return path
    weights = [path for path in root.rglob("*.pth") if path.name in {"model_xd.pth", "model.pth"}]
    if len(weights) == 1:
        return weights[0]
    if not weights:
        archives = [
            path for path in root.rglob("*.zip")
            if any(
                basename(member) in {"model_xd.pth", "model.pth"}
                for member in zip_members(path)
            )
        ]
        if len(archives) == 1:
            return archives[0]
    raise RuntimeError(
        "Gắn checkpoint VadCLIP XD-Violence. Nếu tên khác hoặc có nhiều bản, "
        "đặt CHECKPOINT bằng đường dẫn file .pth hoặc ZIP chỉ chứa một .pth."
    )


def stage_member(archive, member, target):
    # Chỉ ghi tới tên đích do notebook tạo. Không giải nén theo đường dẫn trong ZIP.
    target.parent.mkdir(parents=True, exist_ok=True)
    with archive.open(member) as src, target.open("wb") as dest:
        shutil.copyfileobj(src, dest, length=1024 * 1024)
    return target


def checkpoint_from_input(root, staging):
    path = pick_checkpoint(root)
    if path.suffix.lower() == ".zip":
        members = [name for name in zip_members(path) if name.lower().endswith(".pth")]
        if len(members) != 1:
            raise ValueError(f"ZIP checkpoint phải chứa đúng một .pth: {path.name}")
        if basename(members[0]) == "model_ucf.pth":
            raise ValueError("ZIP đang chứa checkpoint UCF-Crime.")
        with ZipFile(path) as archive:
            path = stage_member(archive, members[0], staging / "checkpoint.pth")
    if path.name == "model_ucf.pth" or path.stat().st_size == 0:
        raise ValueError(f"File checkpoint không hợp lệ cho XD: {path.name}")
    with path.open("rb") as stream:
        if stream.read(4) == b"PK\x03\x04":
            zip_members(path)  # xác nhận có thư mục ZIP ở cuối file PyTorch
    return path


def feature_info(path):
    array = np.load(path, mmap_mode="r", allow_pickle=False)
    if (
        array.ndim != 2 or array.shape[0] == 0 or array.shape[1] != FEATURE_DIM
        or array.dtype not in (np.float16, np.float32, np.float64)
        or not np.isfinite(array).all()
    ):
        raise ValueError(f"Feature cần mảng số thực hữu hạn [T, {FEATURE_DIM}]: {path.name}")
    return len(array), sha256(path)


def read_pack_manifest(root):
    candidates = [path.read_bytes() for path in root.rglob(PACK_MANIFEST_NAME)]
    archives = set(root.rglob(PACK_ARCHIVE_NAME)) | set(root.rglob(PREVIOUS_PACK_ARCHIVE_NAME))
    if FEATURE_ZIP is not None:
        archives.add(Path(FEATURE_ZIP))
    for archive_path in sorted(archives):
        with ZipFile(archive_path) as archive:
            members = [
                name for name in archive.namelist() if basename(name) == PACK_MANIFEST_NAME
            ]
            candidates.extend(archive.read(name) for name in members)
    if not candidates:
        raise FileNotFoundError(
            "Chưa thấy manifest. Gắn Dataset VadCLIP XD-Violence Test 800 "
            f"từ {PACK_ARCHIVE_NAME} và giữ nguyên tên file ZIP."
        )
    if any(hashlib.sha256(raw).hexdigest() != PACK_MANIFEST_SHA256 for raw in candidates):
        raise ValueError("Manifest trên Kaggle không khớp bản đã khóa trong notebook.")

    manifest = json.loads(candidates[0])
    if manifest["upstream_commit"] != COMMIT:
        raise ValueError("Commit trong manifest không đúng.")
    return manifest


def check_pack_manifest(manifest, inventory, checkpoint_hash):
    if manifest["checkpoint_sha256"] != checkpoint_hash:
        raise ValueError("Sai checkpoint so với manifest đã khóa.")
    actual_features = {
        basename(row["official_path"]): {"steps": row["steps"], "sha256": row["feature_sha256"]}
        for row in inventory
    }
    if actual_features != manifest["features"]:
        names = actual_features.keys() | manifest["features"].keys()
        first_difference = next(
            name for name in sorted(names)
            if actual_features.get(name) != manifest["features"].get(name)
        )
        raise ValueError(f"Feature trên Kaggle lệch bản đã khóa: {first_difference}")
    return manifest["source_features_sha256"]


def prepare_inputs(split, gt_path, data_root, staging, upload_names=None):
    fields, rows = read_split(split)
    original_names = {basename(row["path"]) for row in rows}
    if upload_names is not None and (
        len(original_names) != VIDEO_COUNT
        or upload_names.keys() != original_names
        or len(set(upload_names.values())) != VIDEO_COUNT
    ):
        raise ValueError("Ánh xạ tên file không khớp danh sách video trong CSV gốc.")
    names = upload_names if upload_names is not None else {name: name for name in original_names}
    found = matching_files(data_root, set(names.values()))
    resolved = []
    issues = {"missing": [], "ambiguous": []}
    for row in rows:
        original = row["path"]
        candidates = match_one(original, found[names[basename(original)]])
        if len(candidates) == 0:
            issues["missing"].append(original)
        elif len(candidates) > 1:
            issues["ambiguous"].append({
                "expected": original,
                "candidates": [str(path) if member is None else f"{path.name}::{member}"
                               for path, member in candidates],
            })
        else:
            resolved.append(candidates[0])
    if any(issues.values()) or len(set(resolved)) != VIDEO_COUNT:
        report = OUTPUT / "input_issues.json"
        report.write_text(json.dumps(issues, ensure_ascii=False, indent=2), encoding="utf-8")
        raise RuntimeError(
            f"Thiếu {len(issues['missing'])} file, trùng {len(issues['ambiguous'])} dòng "
            f"hoặc dùng lại cùng feature. Xem {report}. Kiểm DATA_ROOT/FEATURE_ZIP."
        )

    gt = np.load(gt_path, allow_pickle=False)
    if gt.shape != (FRAME_COUNT,) or not np.array_equal(np.unique(gt), [0, 1]):
        raise ValueError("Ground truth XD sai shape hoặc không phải vector nhị phân.")

    inventory = []
    with ExitStack() as stack:
        archives = {}
        for index, (row, (path, member)) in enumerate(zip(rows, resolved, strict=True)):
            if member is not None:
                if path not in archives:
                    archives[path] = stack.enter_context(ZipFile(path))
                target = staging / f"{index:04d}" / basename(member)
                path = stage_member(archives[path], member, target)
            count, digest = feature_info(path)
            inventory.append({
                "official_path": row["path"], "label": row["label"],
                "steps": count, "feature_sha256": digest,
            })
            row["path"] = str(path)
    steps = sum(item["steps"] for item in inventory)
    if steps * FRAMES_PER_FEATURE != len(gt):
        raise ValueError(
            f"Feature có {steps:,} bước, cần {len(gt) // FRAMES_PER_FEATURE:,} "
            f"theo ground truth tác giả. Kiểm phiên bản gói feature và crop."
        )

    offset = 0
    for item in inventory:
        stop = offset + item["steps"] * FRAMES_PER_FEATURE
        positive_frames = int(np.count_nonzero(gt[offset:stop]))
        if "_label_A" in basename(item["official_path"]) and positive_frames:
            raise ValueError(f"Lệch ground truth trong video Normal: {item['official_path']}")
        item["positive_frames"] = positive_frames
        offset = stop

    resolved_csv = staging / "xd_CLIP_rgbtest_kaggle.csv"
    with resolved_csv.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return resolved_csv, inventory, gt

# Preflight CPU. Mỗi lần chạy tạo thư mục mới, không ghi đè input đã gắn.
stage_directory = tempfile.TemporaryDirectory(prefix="run_", dir=scratch)
staging = Path(stage_directory.name)
split = src_repo / "list" / "xd_CLIP_rgbtest.csv"
gt_path = src_repo / "list" / "gt.npy"
data_root = Path(DATA_ROOT) if DATA_ROOT is not None else INPUT
try:
    pack_manifest = read_pack_manifest(INPUT) if REQUIRE_PACK_MANIFEST else None
    upload_names = pack_manifest["upload_names"] if pack_manifest is not None else None
    resolved_csv, inventory, gt = prepare_inputs(split, gt_path, data_root, staging, upload_names)
    checkpoint_path = checkpoint_from_input(INPUT, staging)
    checkpoint_hash = sha256(checkpoint_path)
    source_features_hash = None
    if pack_manifest is not None:
        source_features_hash = check_pack_manifest(pack_manifest, inventory, checkpoint_hash)
except Exception:
    stage_directory.cleanup()
    raise

receipt = {
    "upstream_commit": COMMIT,
    "split_sha256": sha256(split),
    "ground_truth_sha256": sha256(gt_path),
    "checkpoint_sha256": checkpoint_hash,
    "pack_manifest_sha256": PACK_MANIFEST_SHA256 if REQUIRE_PACK_MANIFEST else None,
    "source_features_archive_sha256": source_features_hash,
    "video_count": len(inventory),
    "snippet_count": sum(row["steps"] for row in inventory),
    "frame_count": len(gt),
    "frames_per_feature": FRAMES_PER_FEATURE,
    "videos": inventory,
}
if REFERENCE_MANIFEST is not None:
    try:
        reference = json.loads(Path(REFERENCE_MANIFEST).read_text(encoding="utf-8"))
        for key in ("upstream_commit", "split_sha256", "ground_truth_sha256",
                    "checkpoint_sha256", "videos"):
            if receipt[key] != reference.get(key):
                raise ValueError(f"Bản Kaggle lệch bản tham chiếu ở trường {key}.")
    except Exception:
        stage_directory.cleanup()
        raise
receipt["matched_reference_manifest"] = REFERENCE_MANIFEST is not None

(OUTPUT / "preflight.json").write_text(
    json.dumps(receipt, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(
    f"Preflight đạt: {len(inventory)} video, {receipt['snippet_count']:,} bước, "
    f"{len(gt):,} frame."
)
print(f"Checkpoint {checkpoint_path.name}, SHA-256 {checkpoint_hash}")
if REQUIRE_PACK_MANIFEST:
    print("SHA-256: khớp cả 800 feature và checkpoint công bố với hai gói đã chuẩn bị.")

checkpoint_path = Path(EVAL_CHECKPOINT)
evaluated_sha256 = sha256(checkpoint_path)
if evaluated_sha256 != EVAL_SHA256:
    raise ValueError("Checkpoint đã chọn khác hash trong training.json.")
receipt["evaluated_checkpoint_sha256"] = evaluated_sha256

import contextlib
import runpy
import time
import traceback
from unittest.mock import patch




def run_upstream():
    full_run = RUN_MODE == "full"
    packages = {
        "ftfy": "ftfy", "packaging": "packaging", "regex": "regex", "tqdm": "tqdm",
        "sklearn": "scikit-learn", "scipy": "scipy", "pandas": "pandas",
    }
    missing = [
        package for module, package in packages.items()
        if importlib.util.find_spec(module) is None
    ]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

    import torch
    from sklearn import metrics

    if not torch.cuda.is_available():
        raise RuntimeError("Bật Accelerator GPU T4 trong Kaggle để chạy inference.")
    torch.cuda.set_device(0)
    torch.manual_seed(234)
    runtime = staging / "runtime"
    shutil.copytree(src_repo / "src", runtime)
    from v6_runtime_support import duration_diagnostics, install_adapter

    install_adapter(runtime, DILATIONS)
    clip_source = runtime / "clip" / "clip.py"
    text = clip_source.read_text(encoding="utf-8")
    old_import = "from pkg_resources import packaging"
    if text.count(old_import) != 1:
        raise RuntimeError("Không tìm thấy import CLIP cần sửa tương thích tại commit đã khóa.")
    clip_source.write_text(text.replace(old_import, "import packaging.version"), encoding="utf-8")

    if CLIP_WEIGHTS is not None:
        source = Path(CLIP_WEIGHTS)
        if not source.is_file() or sha256(source) != CLIP_SHA256:
            raise ValueError("CLIP_WEIGHTS cần file OpenAI ViT-B-16.pt đúng SHA-256.")
        cache = Path.home() / ".cache" / "clip"
        cache.mkdir(parents=True, exist_ok=True)
        target = cache / "ViT-B-16.pt"
        if source.resolve() != target.resolve():
            with tempfile.TemporaryDirectory(dir=cache) as copy_dir:
                temporary = Path(copy_dir) / target.name
                shutil.copyfile(source, temporary)
                if sha256(temporary) != CLIP_SHA256:
                    raise ValueError("Bản sao trọng số CLIP không khớp SHA-256.")
                temporary.replace(target)

    output_root = OUTPUT / RUN_MODE
    output_root.mkdir(exist_ok=True)
    output_dir = Path(tempfile.mkdtemp(prefix="run_", dir=output_root))
    print(f"Đang chạy evaluator gốc. Log: {output_dir / 'upstream.log'}", flush=True)
    native_ap = metrics.average_precision_score
    ap_calls = []
    upstream_completed = False
    post_ap_error = None
    checkpoint_loaded = False
    forwards = 0
    torch_load = torch.load

    class SmokeComplete(Exception):
        pass

    def load_checkpoint(file, *args, **kwargs):
        nonlocal checkpoint_loaded
        if not isinstance(file, (str, os.PathLike)) or Path(file).resolve() != checkpoint_path.resolve():
            return torch_load(file, *args, **kwargs)
        checkpoint_loaded = True
        state = torch_load(file, map_location="cpu", weights_only=True)
        for field in ("state_dict", "model_state_dict"):
            if isinstance(state, dict) and field in state:
                return state[field]
        return state

    def record_ap(y_true, y_score, *args, **kwargs):
        value = native_ap(y_true, y_score, *args, **kwargs)
        if np.asarray(y_true).shape == gt.shape and np.array_equal(y_true, gt):
            scores = np.asarray(y_score).reshape(-1)
            if scores.shape != gt.shape or not np.isfinite(scores).all():
                raise ValueError("Điểm frame từ evaluator XD sai shape hoặc chứa NaN/Inf.")
            grouped = scores.reshape(-1, FRAMES_PER_FEATURE)
            if not np.all(grouped == grouped[:, :1]):
                raise ValueError("Điểm từ evaluator không theo quy tắc lặp 16 frame.")
            ap_calls.append((float(value), grouped[:, 0].copy()))
        return value

    original_cwd = Path.cwd()
    original_path = sys.path.copy()
    original_argv = sys.argv.copy()
    original_bytecode = sys.dont_write_bytecode
    try:
        os.chdir(runtime)
        sys.path.insert(0, str(runtime))
        sys.dont_write_bytecode = True
        from model import CLIPVAD
        from xd_option import parser

        # Giữ các giá trị mặc định XD của tác giả, chỉ thay đường dẫn input.
        destinations = {action.dest for action in parser._actions}
        overrides = {
            "test_list": str(resolved_csv),
            "gt_path": str(gt_path),
            "model_path": str(checkpoint_path),
        }
        defaults = parser.parse_args([])
        for key in ("gt_segment_path", "gt_label_path"):
            default = Path(getattr(defaults, key))
            path = (src_repo / "src" / default).resolve()
            if not path.is_file():
                path = (src_repo / default).resolve()
            relative = path.relative_to(src_repo)
            git("ls-files", "--error-unmatch", relative.as_posix())
            overrides[key] = str(path)
        if not overrides.keys() <= destinations:
            raise RuntimeError(f"CLI upstream thiếu tham số: {overrides.keys() - destinations}")
        flags = []
        for action in parser._actions:
            if action.dest in overrides:
                flags.extend((action.option_strings[0], overrides[action.dest]))

        original_forward = CLIPVAD.forward

        def count_forward(self, *args, **kwargs):
            nonlocal forwards
            result = original_forward(self, *args, **kwargs)
            forwards += 1
            if RUN_MODE == "smoke" and forwards == 3:
                if any(not torch.isfinite(tensor).all() for tensor in result):
                    raise ValueError("Smoke test sinh NaN/Inf.")
                raise SmokeComplete
            return result

        sys.argv = [str(runtime / "xd_test.py"), *flags]
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        with (
            (output_dir / "upstream.log").open("w", encoding="utf-8") as log,
            contextlib.redirect_stdout(log),
            contextlib.redirect_stderr(log),
            patch.object(torch, "load", load_checkpoint),
            patch.object(metrics, "average_precision_score", record_ap),
            patch.object(CLIPVAD, "forward", count_forward),
        ):
            try:
                runpy.run_path(str(runtime / "xd_test.py"), run_name="__main__")
                upstream_completed = True
            except SmokeComplete:
                if RUN_MODE != "smoke":
                    raise
            except Exception as exc:
                if not full_run or len(ap_calls) != 2:
                    raise
                traceback.print_exc(file=log)
                post_ap_error = f"{type(exc).__name__}: {exc}"
        elapsed = time.perf_counter() - started
    finally:
        os.chdir(original_cwd)
        sys.argv = original_argv
        sys.path[:] = original_path
        sys.dont_write_bytecode = original_bytecode

    if not checkpoint_loaded or forwards == 0:
        raise RuntimeError("Chưa xác nhận upstream đã nạp checkpoint XD và chạy forward.")

    result = {
        **receipt,
        "run_kind": "smoke_inference" if RUN_MODE == "smoke" else "full_checkpoint_inference",
        "runtime_patch": "CLIP imports packaging.version directly",
        "clip_backbone_sha256": CLIP_SHA256,
        "forward_calls": forwards,
        "upstream_completed": upstream_completed,
        "post_ap_error": post_ap_error,
        "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__,
        "elapsed_seconds": elapsed,
        "peak_torch_vram_mib": torch.cuda.max_memory_allocated() / 2**20,
        "metrics": None,
    }
    if full_run:
        if len(ap_calls) != 2:
            raise RuntimeError(
                f"Cần hai AP cấp frame từ xd_test.py, nhận {len(ap_calls)}. "
                f"Xem {output_dir / 'upstream.log'}."
            )
        scores = np.column_stack([column for _, column in ap_calls])
        result["paper_table_1_frame_ap"] = PAPER_AP
        result["metrics"] = {
            f"branch_{index + 1}": {
                "frame_ap": ap,
                "roc_auc": float(metrics.roc_auc_score(gt, np.repeat(column, FRAMES_PER_FEATURE))),
                "gap_vs_paper_percentage_points": 100 * (ap - PAPER_AP),
            }
            for index, (ap, column) in enumerate(ap_calls)
        }
        score_columns = ["branch_1", "branch_2"]
        steps_per_video = [item["steps"] for item in inventory]
        result["duration_diagnostics"] = duration_diagnostics(
            gt, scores[:, 1], steps_per_video, FRAMES_PER_FEATURE,
        )
        result["score_columns"] = score_columns
        np.savez_compressed(
            output_dir / "predictions.npz",
            official_path=np.asarray([item["official_path"] for item in inventory]),
            video_label=np.asarray([item["label"] for item in inventory]),
            snippet_offsets=np.cumsum([0] + steps_per_video),
            score_columns=np.asarray(score_columns),
            scores=scores,
        )
    elif forwards != 3:
        raise RuntimeError("Smoke test chưa đạt ba lượt forward.")

    (output_dir / "run.json").write_text(
        json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    if full_run:
        if post_ap_error is not None:
            print(f"Script gốc lỗi sau khi tính đủ hai AP. Xem upstream.log: {post_ap_error}")
        for name, values in result["metrics"].items():
            print(
                f"{name}: AP {values['frame_ap'] * 100:.4f}% "
                f"(chênh {values['gap_vs_paper_percentage_points']:+.4f} điểm phần trăm)"
            )
    else:
        print("Smoke đạt ba lượt forward. Kiểm đường chạy evaluator XD.")
    print(f"Đầu ra: {output_dir}")


try:
    if RUN_MODE == "preflight":
        print("Đã kiểm input. Chọn GPU và RUN_MODE = 'smoke' để thử checkpoint.")
    else:
        run_upstream()
finally:
    stage_directory.cleanup()
'''


## Huấn luyện, chấm và tổng hợp hai benchmark


In [ ]:
import csv
import json
import math
import os
import subprocess
import sys
import tempfile


def runner_header(config):
    source = ["from pathlib import Path"]
    for key, value in config.items():
        code = f"Path({str(value)!r})" if isinstance(value, Path) else repr(value)
        source.append(f"{key} = {code}")
    return "\n".join(source)


def run_dataset(name, runner, config, work_dir):
    script = work_dir / f"{name}.py"
    script.write_text(
        runner_header(config) + "\n\n" + runner,
        encoding="utf-8",
    )
    console_path = work_dir.parent / f"{name}_console.log"
    print(f"\n--- {name} ---", flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with (
        console_path.open("w", encoding="utf-8") as console,
        subprocess.Popen(
            [sys.executable, "-u", str(script)],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            encoding="utf-8", errors="replace", env=env,
        ) as process,
    ):
        for line in process.stdout:
            console.write(line)
            print(line, end="", flush=True)
        if process.wait() != 0:
            raise RuntimeError(f"Xem log lỗi: {console_path}")
    return console_path



V0_REFERENCE = {
    "ucf": {"roc_auc": 0.8801992605166816, "ano_auc": 0.7023225660564671, "ap": 0.3355725526808031},
    "xd": {"frame_ap": 0.8450448809004643, "roc_auc": 0.946812650099563},
}


def collect_metrics(prefix, reports, training):
    branch = "classification" if prefix == "ucf" else "branch_2"
    spec = (("ROC-AUC", "roc_auc"), ("Ano-AUC", "ano_auc"), ("Frame AP", "ap")) if prefix == "ucf" else (
        ("Frame AP", "frame_ap"), ("ROC-AUC", "roc_auc"),
    )
    split_count = 290 if prefix == "ucf" else 800
    for variant, report in reports.items():
        if report["run_kind"] != "full_checkpoint_inference" or report["video_count"] != split_count:
            raise ValueError("Báo cáo test khác split hoặc chế độ full.")
        if report["split_sha256"] != training["test_csv_sha256"]:
            raise ValueError("Split test khác danh sách đã kiểm khi train.")
        if report["evaluated_checkpoint_sha256"] != training["variants"][variant]["checkpoint_sha256"]:
            raise ValueError("Checkpoint chấm khác checkpoint đã chọn trên development.")
    rows = []
    for label, key in spec:
        control, single, v6 = (
            reports[variant]["metrics"][branch][key] for variant in VARIANTS
        )
        if not all(math.isfinite(value) and 0 <= value <= 1 for value in (control, single, v6)):
            raise ValueError(f"Metric ngoài [0, 1]: {label}")
        v0 = V0_REFERENCE[prefix][key]
        rows.append({
            "dataset": "UCF-Crime" if prefix == "ucf" else "XD-Violence",
            "metric": label, "v0_reference_pct": 100 * v0, "trained_control_pct": 100 * control,
            "single_scale_pct": 100 * single, "v6_pct": 100 * v6,
            "single_vs_control_pp": 100 * (single - control),
            "v6_vs_single_pp": 100 * (v6 - single), "v6_vs_control_pp": 100 * (v6 - control),
            "v6_vs_v0_pp": 100 * (v6 - v0),
        })
    return rows


def find_report(base, prefix, variant, mode):
    run_dir = base / variant / mode
    paths = ([run_dir / "run.json"] if prefix == "ucf"
             else sorted(run_dir.glob("run_*/run.json")))
    paths = [path for path in paths if path.is_file()]
    if len(paths) > 1:
        raise ValueError(f"Có nhiều run.json cho {prefix}/{variant} trong {run_dir}.")
    return paths[0] if paths else None


def run_both():
    if RUN_MODE not in {"smoke", "full"} or EPOCHS < 1 or HALF_BATCH < 1:
        raise ValueError("Cấu hình RUN_MODE, EPOCHS hoặc HALF_BATCH sai.")
    if not INPUT.is_dir():
        raise FileNotFoundError("Gắn sáu Dataset vào notebook qua Add Input.")
    if not DATASETS or len(set(DATASETS)) != len(DATASETS) or set(DATASETS) - {"ucf", "xd"}:
        raise ValueError("DATASETS cần ucf, xd hoặc cả hai, không lặp.")
    previous = Path(RESUME_FROM).resolve() if RESUME_FROM is not None else None
    if previous is not None and (RUN_MODE != "full" or not previous.is_dir()):
        raise ValueError("RESUME_FROM cần thư mục run_* còn tồn tại và RUN_MODE = full.")
    root = WORK / "vadclip_v6"
    root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="run_", dir=root))
    print("Đầu ra chung:", output, flush=True)
    scripts = output / "runtime_scripts"
    scripts.mkdir()
    (scripts / "v6_runtime_support.py").write_text(SUPPORT_SOURCE, encoding="utf-8")
    shared = {
        "INPUT": INPUT, "WORK": WORK, "CLIP_WEIGHTS": CLIP_WEIGHTS,
        "SRC_REPO": Path(tempfile.gettempdir()) / "vadclip_xd_c41067f" / "upstream",
    }
    jobs = (
        ("ucf", UCF_TRAIN_ROOT, UCF_RUNNER, {"DATA_ROOT": UCF_DATA_ROOT, "CHECKPOINT": UCF_CHECKPOINT}),
        ("xd", XD_TRAIN_ROOT, XD_RUNNER, {
            "DATA_ROOT": XD_DATA_ROOT, "CHECKPOINT": XD_CHECKPOINT, "FEATURE_ZIP": XD_FEATURE_ZIP,
            "REFERENCE_MANIFEST": None, "REQUIRE_PACK_MANIFEST": True, "RUN_MODE": RUN_MODE,
        }),
    )
    summaries = {}
    rows = []
    for prefix, train_root, runner, data_config in jobs:
        if prefix not in DATASETS:
            continue
        base = output / prefix
        train_output = (previous / prefix if previous is not None else base) / "training"
        entry = {"status": "running", "test": {}}
        summaries[prefix] = entry
        try:
            if previous is None:
                log = run_dataset(f"{prefix}_train", TRAINER_SOURCE, {
                    **shared, "OUTPUT": train_output, "PREFIX": prefix, "TRAIN_ROOT": train_root,
                    "MANIFEST_SHA256": TRAIN_MANIFESTS[prefix], "SEED": SEED,
                    "EPOCHS": EPOCHS, "HALF_BATCH": HALF_BATCH, "VARIANTS": VARIANTS,
                    "SMOKE_ONLY": RUN_MODE == "smoke",
                }, scripts)
                entry["training_log"] = str(log)
            training_path = train_output / "training.json"
            training = json.loads(training_path.read_text(encoding="utf-8"))
            if training["mode"] != RUN_MODE or set(training["variants"]) != set(VARIANTS):
                raise ValueError("Báo cáo train thiếu cấu hình hoặc sai chế độ.")
            entry["training_json"] = str(training_path)
            reports = {}
            for variant, dilations in VARIANTS.items():
                variant_training = training["variants"][variant]
                if tuple(variant_training["dilations"]) != dilations:
                    raise ValueError(f"Dilation của checkpoint không khớp: {prefix}/{variant}.")
                checkpoint_sha = variant_training["checkpoint_sha256"]
                candidate = None if previous is None else find_report(
                    previous / prefix, prefix, variant, RUN_MODE,
                )
                reused = candidate is not None
                log = None
                if reused:
                    print(f"Dùng lại báo cáo đã hoàn tất: {prefix}/{variant}", flush=True)
                else:
                    checkpoint = train_output / f"{variant}.pth"
                    if not checkpoint.is_file():
                        raise FileNotFoundError(f"Thiếu checkpoint cần chấm: {checkpoint}")
                    log = run_dataset(f"{prefix}_{variant}_test", runner, {
                        **shared, **data_config, "OUTPUT": base / variant,
                        "SMOKE_ONLY": RUN_MODE == "smoke", "DILATIONS": dilations,
                        "EVAL_CHECKPOINT": checkpoint, "EVAL_SHA256": checkpoint_sha,
                    }, scripts)
                    candidate = find_report(base, prefix, variant, RUN_MODE)
                    if candidate is None:
                        raise ValueError(f"Thiếu run.json của {prefix}/{variant}.")
                report = json.loads(candidate.read_text(encoding="utf-8"))
                if report["evaluated_checkpoint_sha256"] != checkpoint_sha:
                    raise ValueError(f"Checkpoint chấm sai hash: {prefix}/{variant}.")
                reports[variant] = report
                entry["test"][variant] = {
                    "run_json": str(candidate), "log": str(log) if log is not None else None,
                    "reused": reused,
                    "duration_diagnostics": report.get("duration_diagnostics"),
                }
            if RUN_MODE == "full":
                rows.extend(collect_metrics(prefix, reports, training))
            entry["status"] = "ok"
        except (OSError, RuntimeError, KeyError, TypeError, ValueError) as exc:
            entry.update(status="failed", error=str(exc))
            print(f"{prefix} lỗi: {exc}", flush=True)

    summary = output / "summary.json"
    summary.write_text(json.dumps({
        "mode": RUN_MODE, "runs": summaries, "metrics": rows,
        "resumed_from": str(previous) if previous is not None else None,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    if rows:
        with (output / "comparison.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        print(f"\n{'Benchmark':<13} {'Metric':<10} {'V0 ref':>10} {'Control':>10} "
              f"{'Single':>10} {'V6':>10} {'Δ V6/Single':>13} {'Δ V6/Control':>14}")
        for row in rows:
            print(f"{row['dataset']:<13} {row['metric']:<10} {row['v0_reference_pct']:>9.4f}% "
                  f"{row['trained_control_pct']:>9.4f}% {row['single_scale_pct']:>9.4f}% "
                  f"{row['v6_pct']:>9.4f}% {row['v6_vs_single_pp']:>+10.4f} pp "
                  f"{row['v6_vs_control_pp']:>+11.4f} pp")
    print("Đầu ra chung:", output)
    if any(entry["status"] != "ok" for entry in summaries.values()):
        raise RuntimeError(f"Có lượt chưa hoàn tất. Xem {summary}.")


run_both()
